In [4]:
import pandas as pd

learn_data = pd.read_csv("preprocess_train_v4.csv", header = None)
learn_data.columns = ['Age', 'DBRatio', 'ALBIScore', 'Glob', 'LogTB', 'LogAlkphos', 'LogSgpt', 'LogSgot', 'Target']
learn_data["Target"] = learn_data["Target"].astype("category")
learn_data.head()

,Age,DBRatio,ALBIScore,Glob,LogTB,LogAlkphos,LogSgpt,LogSgot,Target
0,48,0.511111,0.226640,4.615385,1.504077,5.641907,2.564949,4.304065,0
1,39,0.473684,-0.182383,3.115942,0.641854,5.192957,3.737670,4.127134,0
2,23,0.300000,-0.264120,3.100000,0.000000,5.356586,3.713572,4.382027,0
3,42,0.285714,-0.374875,3.018868,-0.356675,5.023881,3.555348,4.394449,0
4,54,0.504425,0.604032,4.250000,3.117950,6.324359,3.401197,3.610918,0


In [6]:
from sklearn.model_selection import train_test_split

X = learn_data.drop(columns = ["Target"])
y = learn_data["Target"]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size = 0.20, random_state = 42)

## Metrics

In [8]:
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score
import numpy as np

def compute_metrics (y_real, y_pred) -> list[float]:
    F1_macro = f1_score(y_real, y_pred, average = "macro")
    recall = recall_score(y_real, y_pred, average = "macro")
    prec = precision_score(y_real, y_pred, average = "macro")
    acc = accuracy_score(y_real, y_pred)
    return [F1_macro, recall, prec, acc]

def confusion (y_real, y_pred) -> None:
    TP = sum(np.logical_and(y_real == y_pred, y_real == 1))
    TN = sum(np.logical_and(y_real == y_pred, y_real == 0))
    FP = sum(np.logical_and(y_real != y_pred, y_real == 0))
    FN = sum(np.logical_and(y_real != y_pred, y_real == 1))
    print("\t\tPredicted")
    print("\t\t+1\t0")
    print(f"Real\t+1\t{TP}\t{FN}")
    print(f"\t0\t{FP}\t{TN}")
    print(f"Accuracy: {((TP + TN) / y_real.shape[0] * 100):.2f}%".format())

crossval_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])
validation_df = pd.DataFrame(columns = ["F1 Macro", "Recall", "Precision", "Accuracy"])

# KNN

In [9]:
from sklearn.neighbors import KNeighborsClassifier

knc = KNeighborsClassifier(n_neighbors = 5, weights = 'distance')
knc.fit(X_train, y_train)

confusion(np.array(y_train), pd.Series(knc.predict(X_train)))

		Predicted
		+1	0
Real	+1	99	0
	0	0	261
Accuracy: 100.00%


In [10]:
confusion(np.array(y_val), pd.Series(knc.predict(X_val)))

		Predicted
		+1	0
Real	+1	8	21
	0	10	51
Accuracy: 65.56%


In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

knc = KNeighborsClassifier(metric = "minkowski", p = 3)
knc_pipeline = Pipeline([("scaler", StandardScaler()), ("knc", knc)])

n_neighbors = [2, 3, 5, 7, 10, 15]
distances = ["minkowski", "manhattan", "euclidean"]

knc_search = GridSearchCV(estimator = knc_pipeline,
                          param_grid = {"knc__n_neighbors" : n_neighbors,
                                        "knc__weights" : ("uniform", "distance"),
                                        "knc__metric" : distances},
                          scoring = "f1_macro",
                          cv = 5)
knc_search.fit(X_train, y_train)
knc_search.best_params_

{'knc__metric': 'manhattan', 'knc__n_neighbors': 2, 'knc__weights': 'distance'}

In [12]:
knc_search.best_score_

0.5935960453621651

In [14]:
from sklearn.model_selection import cross_validate

knc_n_neighbors = knc_search.best_params_["knc__n_neighbors"]
knc_metric = knc_search.best_params_["knc__metric"]
knc_weights = knc_search.best_params_["knc__weights"]
knc_best = KNeighborsClassifier(n_neighbors = knc_n_neighbors,
                                metric = knc_metric, p = 3,
                                weights = knc_weights)
knc_pipeline = Pipeline([("scaler", StandardScaler()), ("knc", knc_best)])

cross_val_results = pd.DataFrame(cross_validate(knc_pipeline, X_train, y_train, cv = 5, scoring = ['f1_macro', 'recall_macro', 'precision_macro', 'accuracy']))
mean_results = cross_val_results[['test_f1_macro', 'test_recall_macro', 'test_precision_macro', 'test_accuracy']].mean().values

crossval_df.loc["KNN", :] = mean_results
crossval_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
KNN,0.593596,0.592314,0.601528,0.686111


In [17]:
knc_pipeline.fit(X_train, y_train)
validation_df.loc["KNN", :] = compute_metrics(y_val, knc_pipeline.predict(X_val))
validation_df.sort_values(by = "F1 Macro", ascending = False)

,F1 Macro,Recall,Precision,Accuracy
KNN,0.630682,0.624081,0.662713,0.711111
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN


In [18]:
confusion(y_val, knc_pipeline.predict(X_val))

		Predicted
		+1	0
Real	+1	11	18
	0	8	53
Accuracy: 71.11%


## Predictions

In [19]:
test_data = pd.read_csv("preprocess_test_v4.csv", header = None)
test_data.columns = ['Age', 'DBRatio', 'ALBIScore', 'Glob', 'LogTB', 'LogAlkphos', 'LogSgpt', 'LogSgot']
test_data.head()

,Age,DBRatio,ALBIScore,Glob,LogTB,LogAlkphos,LogSgpt,LogSgot
0,11,0.142857,-0.460075,3.000000,-0.356675,6.383507,3.258097,3.367296
1,62,0.500000,-0.172320,5.000000,0.587787,5.411646,4.234107,5.043425
2,60,0.285714,-0.460075,3.818182,-0.356675,5.159055,3.465736,2.639057
3,60,0.491228,0.226237,4.102564,1.740466,5.365976,6.021023,6.745236
4,48,0.222222,-0.260240,3.000000,-0.105361,5.164786,3.178054,3.988984


In [20]:
knc_pipeline.fit(X, y)

labels_knc = pd.DataFrame(columns = ['ID', 'Label'])
labels_knc['Label'] = pd.DataFrame(knc_pipeline.predict(test_data))
labels_knc['ID'] = labels_knc.index + 1
labels_knc.to_csv('new_predictions/knn_best_fs.csv', index = False)